# Assignment 5 - Part 2: Sentiment Analysis with BERT

Fine-tune a pre-trained DistilBERT model to classify text as positive, neutral, or negative.

Dataset: https://www.kaggle.com/datasets/abhi8923shriv/sentiment-analysis-dataset

Download the CSV and place `train.csv` next to this notebook.

In [1]:
# !pip install transformers torch pandas scikit-learn

In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

/Users/codula/Documents/CMI/AML/AppliedMachineLearning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Using device: cpu


## Load and prepare data

Keep only the text and sentiment columns, drop missing rows, and map labels to integers.

In [3]:
df = pd.read_csv('train.csv', encoding='latin-1')
df = df[['text', 'sentiment']].dropna()

# Optional: take a sample to train faster
df = df.sample(n=5000, random_state=42).reset_index(drop=True)

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_map)
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(df['sentiment'].value_counts())

sentiment
neutral     2040
positive    1521
negative    1439
Name: count, dtype: int64


In [4]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('Train:', len(train_df), '| Val:', len(val_df))

Train: 4000 | Val: 1000


## Tokenizer and Dataset

Use DistilBERT's tokenizer (faster than BERT, similar accuracy).

In [5]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, max_len=64):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = SentimentDataset(train_df['text'].tolist(), train_df['label'].tolist())
val_ds = SentimentDataset(val_df['text'].tolist(), val_df['label'].tolist())

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

## Load pre-trained DistilBERT with a classification head

In [6]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Train

In [7]:
epochs = 2
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer.step()
        total_loss += outputs.loss.item()
    print(f'Epoch {epoch+1}/{epochs} - loss: {total_loss/len(train_loader):.4f}')

Epoch 1/2 - loss: 0.7569


Epoch 2/2 - loss: 0.4755


## Evaluate and print classification report

In [8]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']

        outputs = model(input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        y_true.extend(labels.numpy())
        y_pred.extend(preds)

target_names = ['negative', 'neutral', 'positive']
print(classification_report(y_true, y_pred, target_names=target_names))

              precision    recall  f1-score   support

    negative       0.76      0.82      0.79       288
     neutral       0.77      0.71      0.74       408
    positive       0.81      0.84      0.82       304

    accuracy                           0.78      1000
   macro avg       0.78      0.79      0.78      1000
weighted avg       0.78      0.78      0.78      1000

